In [1]:
import sys
from pathlib import Path

def find_data_folder():
    """DATA 폴더 찾기"""
    current = Path.cwd()

    # 상위 5단계까지 검색
    for i in range(5):
        data_path = current / "DATA"
        if data_path.exists():
            target_files = [
                data_path / "stock_invest_function.py",
                data_path / "us_stock_valuation_preprocessing.py"
            ]
            if all(f.exists() for f in target_files):
                return data_path
        current = current.parent
    return None

def fix_and_import_modules():
    """모듈을 수정해서 import"""
    data_path = find_data_folder()
    if not data_path:
        print("DATA 폴더를 찾을 수 없습니다")
        return None, None

    sys.path.insert(0, str(data_path))
    print(f"DATA 폴더 발견: {data_path}")

    # stock_invest_function은 정상 import
    try:
        import stock_invest_function as stock_func
        print("stock_invest_function import 성공!")
    except Exception as e:
        print(f"stock_invest_function import 실패: {e}")
        stock_func = None

    # us_stock_valuation_preprocessing은 파일을 읽어서 수정 후 실행
    try:
        preprocessing_file = data_path / "us_stock_valuation_preprocessing.py"

        # 파일 내용 읽기
        with open(preprocessing_file, 'r', encoding='utf-8') as f:
            content = f.read()

        # 문제가 되는 부분을 찾아서 수정
        # "from stock_invest_function import *"를 모듈 최상위로 이동
        lines = content.split('\n')

        # import * 구문을 찾아서 제거하고 최상위에 추가
        fixed_lines = []
        import_lines = []
        inside_function = False

        for line in lines:
            # 함수 시작 감지
            if line.strip().startswith('def '):
                inside_function = True

            # import * 구문 처리
            if 'from stock_invest_function import *' in line:
                if inside_function:
                    # 함수 내부에 있으면 제거
                    continue
                else:
                    # 최상위에 있으면 그대로 유지
                    import_lines.append(line)
                    continue

            fixed_lines.append(line)

        # 수정된 내용 조합
        final_content = '\n'.join(import_lines + [''] + fixed_lines)

        # 수정된 내용을 실행해서 모듈 생성
        import types
        stock_prep = types.ModuleType('us_stock_valuation_preprocessing')

        # get_db_host 함수가 필요한 경우를 위해 미리 정의
        if stock_func and hasattr(stock_func, 'get_db_host'):
            stock_prep.get_db_host = stock_func.get_db_host
        else:
            stock_prep.get_db_host = lambda: 'localhost'

        # 수정된 코드 실행
        exec(final_content, stock_prep.__dict__)

        print("us_stock_valuation_preprocessing 수정 후 import 성공!")
        return stock_func, stock_prep

    except Exception as e:
        print(f"파일 수정 중 오류: {e}")
        return stock_func, None

# 실행
stock_func, stock_prep = fix_and_import_modules()

if stock_func and stock_prep:
    print("모든 모듈이 성공적으로 로드되었습니다!")
    print("사용 가능한 함수들:")
    if hasattr(stock_prep, 'run_data_preprocessing'):
        print("- stock_prep.run_data_preprocessing()")
    if hasattr(stock_prep, 'run_simple_preprocessing'):
        print("- stock_prep.run_simple_preprocessing()")
else:
    print("모듈 로드에 실패했습니다.")

DATA 폴더 발견: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
stock_invest_function import 성공!
us_stock_valuation_preprocessing 수정 후 import 성공!
모든 모듈이 성공적으로 로드되었습니다!
사용 가능한 함수들:
- stock_prep.run_data_preprocessing()
- stock_prep.run_simple_preprocessing()


In [2]:
from DATA.stock_invest_function import *

In [3]:
api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

In [38]:
# 간단 사용
# result = stock_prep.run_data_preprocessing(['SMCI'], api_key)

# 수출 데이터 포함
result = stock_prep.run_data_preprocessing(['AMAT'], api_key, '848690', db_info)

# 단일 종목 간단 버전
# result = stock_prep.run_simple_preprocessing('AAPL', api_key)

데이터 전처리 시작
대상 종목: AMAT
HS Code: 848690
API 연결 테스트 중...
API 연결 성공
수출 데이터 수집 중... (HS Code: 848690)
가장 최근 input_date: 2025-09-02
해당 날짜의 데이터: 165개
수출 데이터 수집 완료: 165 레코드
매출 데이터 수집 중...


Revenue:   0%|          | 0/1 [00:00<?, ?it/s]

   AMAT: 160개 분기


Revenue: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]


매출 데이터 수집 완료: 160 레코드
시가총액 데이터 수집 중...


Market Cap: 100%|██████████| 1/1 [00:20<00:00, 20.53s/it]

   AMAT: 189개 월
시가총액 데이터 수집 완료: 189 레코드
매출 및 시가총액 데이터 결합 중...
데이터 결합 완료: 188 레코드
PSR 계산 중...
PSR 계산 완료: 186 레코드 (제거된 레코드: 2개)
수출 데이터와 결합 중...
최종 데이터 결합 완료: 200 레코드
   수출 데이터: 166개
   시가총액 데이터: 186개
   수출 예측치 포함: -20개
데이터 전처리 완료!
최종 데이터: 200 레코드
수출 예측치만 있는 미래 데이터: 14 레코드


In [39]:
from us_sarima_forecast import *

In [40]:
# 1. 각 모델별로 구분된 변수명 사용
# =================================================================

# SARIMA 외생변수 포함 예측
sarima_forecast_result = sarima_forecast_with_export(
    final_data=result,
    export_forecast_start_date="2025-10",
    USE_EXOGENOUS=True,
    forecast_months=12
)

# SARIMA 매출 예측 파이프라인
sarima_result_data, sarima_quarterly_data, sarima_forecast_result, sarima_model_info = revenue_sarima_forecast_pipeline(
    data=result,
    revenue_col='revenue_billions',
    date_col='date_month_end',
    data_end_date='2025-08-31',
    forecast_quarters=4
)

# LSTM 매출 예측 파이프라인
lstm_result_data, lstm_quarterly_data, lstm_forecast_result, lstm_model_info = revenue_lstm_forecast_pipeline(
    data=result,
    revenue_col='revenue_billions',
    date_col='date_month_end',
    data_end_date='2025-08-31',
    forecast_quarters=4,
    lookback_window=8,
    epochs=100
)

# Prophet 매출 예측 파이프라인
prophet_result_data, prophet_quarterly_data, prophet_forecast_results, prophet_model_infos = revenue_prophet_forecast_pipeline(
    data=result,
    revenue_col='revenue_billions',
    date_col='date_month_end',
    data_end_date='2025-08-31',
    forecast_quarters=4,
    use_exogenous=True,
    exog_cols=['expDlr']
)

# Exponential Smoothing 매출 예측 파이프라인
es_result_data, es_quarterly_data, es_forecast_result, es_model_info = revenue_es_forecast_pipeline(
    data=result,
    revenue_col='revenue_billions',
    date_col='date_month_end',
    data_end_date='2025-08-31',
    forecast_quarters=4
)

# 2. 모든 결과를 딕셔너리로 통합 관리
# =================================================================

# 모델별 결과 통합
all_forecast_results = {
    'sarima': {
        'result_data': sarima_result_data,
        'quarterly_data': sarima_quarterly_data,
        'forecast_result': sarima_forecast_result,
        'model_info': sarima_model_info
    },
    'lstm': {
        'result_data': lstm_result_data,
        'quarterly_data': lstm_quarterly_data,
        'forecast_result': lstm_forecast_result,
        'model_info': lstm_model_info
    },
    'prophet': {
        'result_data': prophet_result_data,
        'quarterly_data': prophet_quarterly_data,
        'forecast_result': prophet_forecast_results,
        'model_info': prophet_model_infos
    },
    'exponential_smoothing': {
        'result_data': es_result_data,
        'quarterly_data': es_quarterly_data,
        'forecast_result': es_forecast_result,
        'model_info': es_model_info
    }
}

# 3. 모든 예측 결과를 하나의 DataFrame으로 통합
# =================================================================

import pandas as pd
from datetime import datetime

def combine_all_forecasts(forecast_results_dict):
    """
    모든 모델의 예측 결과를 하나의 DataFrame으로 통합
    """
    combined_forecasts = []

    for model_name, results in forecast_results_dict.items():
        try:
            forecast_data = results['forecast_result']

            # forecast_result가 DataFrame인 경우
            if isinstance(forecast_data, pd.DataFrame):
                temp_df = forecast_data.copy()
                temp_df['model'] = model_name
                combined_forecasts.append(temp_df)

            # forecast_result가 다른 형태인 경우 (딕셔너리 등)
            elif isinstance(forecast_data, dict):
                # 예측값이 포함된 키를 찾아서 처리
                if 'forecast' in forecast_data:
                    temp_df = pd.DataFrame(forecast_data['forecast'])
                    temp_df['model'] = model_name
                    combined_forecasts.append(temp_df)

        except Exception as e:
            print(f"모델 {model_name} 결과 통합 중 오류: {e}")
            continue

    if combined_forecasts:
        return pd.concat(combined_forecasts, ignore_index=True)
    else:
        return pd.DataFrame()

# 통합된 예측 결과 생성
combined_forecast_df = combine_all_forecasts(all_forecast_results)

# 4. 결과 데이터를 컬럼별로 통합하는 방법
# =================================================================

def merge_forecast_columns(forecast_results_dict, base_data):
    """
    각 모델의 예측 결과를 base_data에 새로운 컬럼으로 추가
    """
    result_df = base_data.copy()

    for model_name, results in forecast_results_dict.items():
        try:
            forecast_data = results['forecast_result']

            # 예측 컬럼명 설정
            forecast_col = f'forecast_{model_name}'
            confidence_lower_col = f'forecast_{model_name}_lower'
            confidence_upper_col = f'forecast_{model_name}_upper'

            # forecast_data에서 예측값 추출하여 병합
            if isinstance(forecast_data, pd.DataFrame):
                # 날짜 기준으로 병합
                if 'date' in forecast_data.columns:
                    merge_df = forecast_data[['date', 'forecast']].copy()
                    merge_df.columns = ['date', forecast_col]
                    result_df = pd.merge(result_df, merge_df, on='date', how='left')

                    # 신뢰구간이 있는 경우 추가
                    if 'forecast_lower' in forecast_data.columns:
                        lower_df = forecast_data[['date', 'forecast_lower']].copy()
                        lower_df.columns = ['date', confidence_lower_col]
                        result_df = pd.merge(result_df, lower_df, on='date', how='left')

                    if 'forecast_upper' in forecast_data.columns:
                        upper_df = forecast_data[['date', 'forecast_upper']].copy()
                        upper_df.columns = ['date', confidence_upper_col]
                        result_df = pd.merge(result_df, upper_df, on='date', how='left')

        except Exception as e:
            print(f"모델 {model_name} 컬럼 병합 중 오류: {e}")
            continue

    return result_df

# base_data는 원본 데이터 (result 변수)
final_combined_data = merge_forecast_columns(all_forecast_results, result)

# 5. 모델 성능 비교를 위한 요약 테이블 생성
# =================================================================

def create_model_summary(forecast_results_dict):
    """
    모든 모델의 성능 지표를 요약한 테이블 생성
    """
    summary_data = []

    for model_name, results in forecast_results_dict.items():
        try:
            model_info = results['model_info']

            summary_row = {'model': model_name}

            # 모델 정보에서 성능 지표 추출
            if isinstance(model_info, dict):
                # 일반적인 성능 지표들
                metrics = ['mse', 'rmse', 'mae', 'mape', 'aic', 'bic', 'r2']
                for metric in metrics:
                    if metric in model_info:
                        summary_row[metric] = model_info[metric]

            summary_data.append(summary_row)

        except Exception as e:
            print(f"모델 {model_name} 요약 생성 중 오류: {e}")
            continue

    return pd.DataFrame(summary_data)

# 모델 성능 요약 테이블
model_performance_summary = create_model_summary(all_forecast_results)

# 6. 결과 확인 및 저장
# =================================================================

print("=== 모든 예측 모델 결과 저장 완료 ===")
print(f"SARIMA 결과: sarima_result_data, sarima_forecast_result")
print(f"LSTM 결과: lstm_result_data, lstm_forecast_result")
print(f"Prophet 결과: prophet_result_data, prophet_forecast_results")
print(f"ES 결과: es_result_data, es_forecast_result")
print(f"통합 결과: all_forecast_results (딕셔너리)")
print(f"통합 예측 DataFrame: combined_forecast_df")
print(f"컬럼별 통합 데이터: final_combined_data")
print(f"모델 성능 요약: model_performance_summary")

# 각 모델 결과에 개별적으로 접근하는 방법
print("\n=== 개별 모델 결과 접근 방법 ===")
print("SARIMA 예측 결과:", type(all_forecast_results['sarima']['forecast_result']))
print("LSTM 예측 결과:", type(all_forecast_results['lstm']['forecast_result']))
print("Prophet 예측 결과:", type(all_forecast_results['prophet']['forecast_result']))
print("ES 예측 결과:", type(all_forecast_results['exponential_smoothing']['forecast_result']))

# 7. 선택적 저장 (필요한 경우)
# =================================================================

# CSV 파일로 저장
# combined_forecast_df.to_csv('combined_forecasts.csv', index=False)
# final_combined_data.to_csv('final_combined_data.csv', index=False)
# model_performance_summary.to_csv('model_performance_summary.csv', index=False)

# Excel 파일로 모든 결과 저장
# with pd.ExcelWriter('forecast_results.xlsx') as writer:
#     combined_forecast_df.to_excel(writer, sheet_name='Combined_Forecasts', index=False)
#     final_combined_data.to_excel(writer, sheet_name='Final_Data', index=False)
#     model_performance_summary.to_excel(writer, sheet_name='Model_Summary', index=False)
#
#     # 각 모델별 상세 결과도 별도 시트로 저장
#     for model_name, results in all_forecast_results.items():
#         try:
#             if isinstance(results['forecast_result'], pd.DataFrame):
#                 results['forecast_result'].to_excel(writer, sheet_name=f'{model_name}_forecast', index=False)
#         except:
#             pass

SARIMA 예측 시작
예측 시작일: 2025-10
외생변수 사용: True
예측 기간: 12개월
예측 종료일: 2026-09
과거 수출 데이터: 154개
미래 수출 예측치: 12개
과거 PSR 데이터: 186개

외생변수(수출 데이터) 준비 중... (YoY 변환)
외생변수(YoY) 매칭된 학습 개월: 139
미래 수출 YoY 예측치 개월: 12

PSR 정상성 검정:
ADF Statistic: -2.5515
p-value: 0.1034
시계열이 비정상적입니다. 차분이 필요할 수 있습니다.

수출 데이터 정상성 검정:
ADF Statistic: -2.7666
p-value: 0.0632
시계열이 비정상적입니다. 차분이 필요할 수 있습니다.
최적 SARIMA 파라미터 탐색 중...
총 144개 조합 테스트
최적 파라미터를 찾을 수 없어 기본값을 사용합니다.

최적 파라미터:
ARIMA Order: (1, 1, 1)
Seasonal Order: (1, 1, 1, 12)
Best AIC: 0.0000

최종 SARIMA 모델 학습 중...
모델 학습 실패: endog and exog matrices are different sizes
=== 매출 SARIMA 예측 파이프라인 시작 ===
데이터 종료일: 2025-08-31
예측 분기 수: 4

1. 분기별 매출 데이터 추출
데이터를 2025-08까지로 제한했습니다.
유효한 매출 데이터: 185개월
데이터 기간: 2010-03 ~ 2025-08
추출된 분기 데이터: 63분기
분기별 데이터:
  2010Q1: 1.85B (1개월 데이터)
  2010Q2: 2.30B (3개월 데이터)
  2010Q3: 2.52B (3개월 데이터)
  2010Q4: 2.89B (3개월 데이터)
  2011Q1: 2.69B (3개월 데이터)
  2011Q2: 2.86B (3개월 데이터)
  2011Q3: 2.79B (3개월 데이터)
  2011Q4: 2.18B (3개월 데이터)
  2012Q1: 2.19B (3개월 데이터)
  2012Q2

00:45:30 - cmdstanpy - INFO - Chain [1] start processing


Prophet 모델 훈련 중...


00:45:30 - cmdstanpy - INFO - Chain [1] done processing
00:45:30 - cmdstanpy - INFO - Chain [1] start processing


분기별 Prophet 예측 완료 (외생변수 미포함):
  2025Q4: 4.93B
  2026Q1: 4.69B
  2026Q2: 6.20B
  2026Q3: 6.00B
외생변수 미포함 Prophet 예측 성공
분기별 Prophet 예측값을 월별로 분배 중...
2025Q4 Prophet 예측값 4.93B를 [10, 11, 12]월에 동일하게 적용
  -> 2025-10-31: 4.93B
  -> 2025-11-30: 4.93B
  -> 2025-12-31: 4.93B
2026Q1 Prophet 예측값 4.69B를 [1, 2, 3]월에 동일하게 적용
  -> 2026-01-31: 4.69B
  -> 2026-02-28: 4.69B
  -> 2026-03-31: 4.69B
2026Q2 Prophet 예측값 6.20B를 [4, 5, 6]월에 동일하게 적용
  -> 2026-04-30: 6.20B
  -> 2026-05-31: 6.20B
  -> 2026-06-30: 6.20B
2026Q3 Prophet 예측값 6.00B를 [7, 8, 9]월에 동일하게 적용
  -> 2026-07-31: 6.00B
  -> 2026-08-31: 6.00B
  -> 2026-09-30: 6.00B

3. Prophet 예측 (외생변수 포함)
사용 가능한 외생변수: ['expDlr']
외생변수 NaN 제거: 34개 행 제거됨 (전체 187개 중)
추출된 분기별 외생변수 데이터: 51분기
외생변수 데이터 추가 중...
추가할 외생변수: ['expDlr']
외생변수 expDlr 매핑 실패로 12개 행 제거됨
추가된 외생변수: ['expDlr']
Prophet 데이터 준비 완료: 51개 분기
Prophet 모델링 시작 (외생변수 포함)
외생변수 추가: expDlr
Prophet 모델 훈련 중...


00:45:30 - cmdstanpy - INFO - Chain [1] done processing


외생변수의 미래 값 설정 중...
Prophet 예측 실패: Regressor 'expDlr' missing from dataframe
외생변수 포함 Prophet 예측 실패

=== 매출 Prophet 예측 완료 ===
완료된 예측: 1/2개
추가된 컬럼:
  - revenue_billions_prophet_forecast
=== 매출 Exponential Smoothing 예측 파이프라인 시작 ===
데이터 종료일: 2025-08-31
예측 분기 수: 4

1. 분기별 매출 데이터 추출
데이터를 2025-08까지로 제한했습니다.
유효한 매출 데이터: 185개월
데이터 기간: 2010-03 ~ 2025-08
추출된 분기 데이터: 63분기
분기별 데이터:
  2010Q1: 1.85B (1개월 데이터)
  2010Q2: 2.30B (3개월 데이터)
  2010Q3: 2.52B (3개월 데이터)
  2010Q4: 2.89B (3개월 데이터)
  2011Q1: 2.69B (3개월 데이터)
  2011Q2: 2.86B (3개월 데이터)
  2011Q3: 2.79B (3개월 데이터)
  2011Q4: 2.18B (3개월 데이터)
  2012Q1: 2.19B (3개월 데이터)
  2012Q2: 2.54B (3개월 데이터)
  2012Q3: 2.34B (3개월 데이터)
  2012Q4: 1.65B (3개월 데이터)
  2013Q1: 1.57B (3개월 데이터)
  2013Q2: 1.97B (3개월 데이터)
  2013Q3: 1.98B (3개월 데이터)
  2013Q4: 1.99B (3개월 데이터)
  2014Q1: 2.19B (3개월 데이터)
  2014Q2: 2.35B (3개월 데이터)
  2014Q3: 2.27B (3개월 데이터)
  2014Q4: 2.26B (3개월 데이터)
  2015Q1: 2.36B (3개월 데이터)
  2015Q2: 2.44B (3개월 데이터)
  2015Q3: 2.49B (3개월 데이터)
  2015Q4: 2.37B (3개월 데이터)
  201

Traceback (most recent call last):
  File "C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\us_sarima_forecast.py", line 1378, in prophet_quarterly_forecast
    forecast = model.predict(future_df)
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\prophet\forecaster.py", line 1273, in predict
    df = self.setup_dataframe(df.copy())
  File "C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\prophet\forecaster.py", line 300, in setup_dataframe
    raise ValueError(
ValueError: Regressor 'expDlr' missing from dataframe


In [41]:
model_performance_summary.tail(12)

,model,aic
0,sarima,-6.997880
1,lstm,NaN
2,prophet,NaN
3,exponential_smoothing,-165.890127


In [42]:
combined_forecast_df

,date_quarter_end,year,quarter,year_quarter,revenue_billions_forecast,forecast_lower,forecast_upper,model,revenue_billions_lstm_forecast,revenue_billions_es_forecast
0,2025-12-31,2025,4,2025Q4,7.520902,7.162723,7.879081,sarima,NaN,NaN
1,2026-03-31,2026,1,2026Q1,7.592293,7.234114,7.950472,sarima,NaN,NaN
2,2026-06-30,2026,2,2026Q2,7.575175,7.216996,7.933354,sarima,NaN,NaN
3,2026-09-30,2026,3,2026Q3,7.640748,7.282569,7.998927,sarima,NaN,NaN
4,2025-12-31,2025,4,2025Q4,NaN,NaN,NaN,lstm,7.217980,NaN
5,2026-03-31,2026,1,2026Q1,NaN,NaN,NaN,lstm,7.255886,NaN
6,2026-06-30,2026,2,2026Q2,NaN,NaN,NaN,lstm,7.266075,NaN
7,2026-09-30,2026,3,2026Q3,NaN,NaN,NaN,lstm,7.253175,NaN
8,2025-12-31,2025,4,2025Q4,NaN,6.879944,7.844019,exponential_smoothing,NaN,7.361981
9,2026-03-31,2026,1,2026Q1,NaN,6.929529,7.893604,exponential_smoothing,NaN,7.411567


In [43]:
final_combined_data

,ticker,date_month_end,market_cap_billions,revenue_billions,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr
0,AMAT,2010-03-31,18.11,1.85,5.529113,5.529113,3.275390,NaN,NaN
1,AMAT,2010-04-30,18.53,1.85,5.529113,5.529113,3.351351,NaN,NaN
2,AMAT,2010-05-31,17.31,2.30,6.804576,5.529113,3.130701,NaN,NaN
3,AMAT,2010-06-30,16.10,2.30,6.804576,5.529113,2.911859,NaN,NaN
4,AMAT,2010-07-31,15.81,2.30,6.804576,6.804576,2.323436,NaN,NaN
...,...,...,...,...,...,...,...,...,...
195,AMAT,2026-05-31,NaN,NaN,NaN,NaN,NaN,848690,571528000.0
196,AMAT,2026-06-30,NaN,NaN,NaN,NaN,NaN,848690,579657000.0
197,AMAT,2026-07-31,NaN,NaN,NaN,NaN,NaN,848690,560953000.0
198,AMAT,2026-08-31,NaN,NaN,NaN,NaN,NaN,848690,575292000.0


In [51]:
sarima_result_data

,ticker,date_month_end,market_cap_billions,revenue_billions,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr,revenue_billions_forecast
0,AMAT,2010-03-31,18.11,1.85,5.529113,5.529113,3.275390,NaN,NaN,1.850000
1,AMAT,2010-04-30,18.53,1.85,5.529113,5.529113,3.351351,NaN,NaN,1.850000
2,AMAT,2010-05-31,17.31,2.30,6.804576,5.529113,3.130701,NaN,NaN,2.300000
3,AMAT,2010-06-30,16.10,2.30,6.804576,5.529113,2.911859,NaN,NaN,2.300000
4,AMAT,2010-07-31,15.81,2.30,6.804576,6.804576,2.323436,NaN,NaN,2.300000
...,...,...,...,...,...,...,...,...,...,...
195,AMAT,2026-05-31,NaN,NaN,NaN,NaN,NaN,848690,571528000.0,7.575175
196,AMAT,2026-06-30,NaN,NaN,NaN,NaN,NaN,848690,579657000.0,7.575175
197,AMAT,2026-07-31,NaN,NaN,NaN,NaN,NaN,848690,560953000.0,7.640748
198,AMAT,2026-08-31,NaN,NaN,NaN,NaN,NaN,848690,575292000.0,7.640748


In [46]:
lstm_result_data.tail(10)

,ticker,date_month_end,market_cap_billions,revenue_billions,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr,revenue_billions_lstm_forecast
190,AMAT,2025-12-31,NaN,NaN,NaN,NaN,NaN,848690,537453000.0,7.217980
191,AMAT,2026-01-31,NaN,NaN,NaN,NaN,NaN,848690,540713000.0,7.255886
192,AMAT,2026-02-28,NaN,NaN,NaN,NaN,NaN,848690,510855000.0,7.255886
193,AMAT,2026-03-31,NaN,NaN,NaN,NaN,NaN,848690,602568000.0,7.255886
194,AMAT,2026-04-30,NaN,NaN,NaN,NaN,NaN,848690,607628000.0,7.266075
195,AMAT,2026-05-31,NaN,NaN,NaN,NaN,NaN,848690,571528000.0,7.266075
196,AMAT,2026-06-30,NaN,NaN,NaN,NaN,NaN,848690,579657000.0,7.266075
197,AMAT,2026-07-31,NaN,NaN,NaN,NaN,NaN,848690,560953000.0,7.253175
198,AMAT,2026-08-31,NaN,NaN,NaN,NaN,NaN,848690,575292000.0,7.253175
199,AMAT,2026-09-30,NaN,NaN,NaN,NaN,NaN,848690,620370000.0,7.253175


In [47]:
prophet_result_data.tail(10)

,ticker,date_month_end,market_cap_billions,revenue_billions,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr,revenue_billions_prophet_forecast
190,AMAT,2025-12-31,NaN,NaN,NaN,NaN,NaN,848690,537453000.0,4.925787
191,AMAT,2026-01-31,NaN,NaN,NaN,NaN,NaN,848690,540713000.0,4.689668
192,AMAT,2026-02-28,NaN,NaN,NaN,NaN,NaN,848690,510855000.0,4.689668
193,AMAT,2026-03-31,NaN,NaN,NaN,NaN,NaN,848690,602568000.0,4.689668
194,AMAT,2026-04-30,NaN,NaN,NaN,NaN,NaN,848690,607628000.0,6.198798
195,AMAT,2026-05-31,NaN,NaN,NaN,NaN,NaN,848690,571528000.0,6.198798
196,AMAT,2026-06-30,NaN,NaN,NaN,NaN,NaN,848690,579657000.0,6.198798
197,AMAT,2026-07-31,NaN,NaN,NaN,NaN,NaN,848690,560953000.0,6.001910
198,AMAT,2026-08-31,NaN,NaN,NaN,NaN,NaN,848690,575292000.0,6.001910
199,AMAT,2026-09-30,NaN,NaN,NaN,NaN,NaN,848690,620370000.0,6.001910


In [48]:
es_result_data.tail(10)

,ticker,date_month_end,market_cap_billions,revenue_billions,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr,revenue_billions_es_forecast
190,AMAT,2025-12-31,NaN,NaN,NaN,NaN,NaN,848690,537453000.0,7.361981
191,AMAT,2026-01-31,NaN,NaN,NaN,NaN,NaN,848690,540713000.0,7.411567
192,AMAT,2026-02-28,NaN,NaN,NaN,NaN,NaN,848690,510855000.0,7.411567
193,AMAT,2026-03-31,NaN,NaN,NaN,NaN,NaN,848690,602568000.0,7.411567
194,AMAT,2026-04-30,NaN,NaN,NaN,NaN,NaN,848690,607628000.0,7.451235
195,AMAT,2026-05-31,NaN,NaN,NaN,NaN,NaN,848690,571528000.0,7.451235
196,AMAT,2026-06-30,NaN,NaN,NaN,NaN,NaN,848690,579657000.0,7.451235
197,AMAT,2026-07-31,NaN,NaN,NaN,NaN,NaN,848690,560953000.0,7.482969
198,AMAT,2026-08-31,NaN,NaN,NaN,NaN,NaN,848690,575292000.0,7.482969
199,AMAT,2026-09-30,NaN,NaN,NaN,NaN,NaN,848690,620370000.0,7.482969


In [8]:
# 현재 stock_prep 모듈에 어떤 함수들이 있는지 확인
print("stock_prep 모듈의 속성들:")
attributes = [attr for attr in dir(stock_prep) if not attr.startswith('_')]
for attr in attributes:
    print(f"- {attr}")

# 함수인지 확인
print("\n함수들:")
import types
functions = [attr for attr in dir(stock_prep) if isinstance(getattr(stock_prep, attr), types.FunctionType)]
for func in functions:
    print(f"- {func}()")


stock_prep 모듈의 속성들:
- STOCK_FUNCTION_AVAILABLE
- add_revenue_ttm
- calculate_psr_with_shift
- calendar
- collect_export_data
- collect_market_cap_data
- collect_revenue_data
- convert_to_month_end
- create_engine
- datetime
- fetch_market_data_yearly
- fetch_revenue_data
- get_db_host
- get_hs_data
- get_latest_input_date_data
- merge_revenue_market_data
- merge_with_export_data
- pd
- process_daily_to_monthly_market_data
- requests
- run_data_preprocessing
- run_simple_preprocessing
- test_api_connection
- time
- tqdm
- warnings

함수들:
- add_revenue_ttm()
- calculate_psr_with_shift()
- collect_export_data()
- collect_market_cap_data()
- collect_revenue_data()
- convert_to_month_end()
- create_engine()
- fetch_market_data_yearly()
- fetch_revenue_data()
- get_db_host()
- get_hs_data()
- get_latest_input_date_data()
- merge_revenue_market_data()
- merge_with_export_data()
- process_daily_to_monthly_market_data()
- run_data_preprocessing()
- run_simple_preprocessing()
- test_api_connectio